In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
V = {}
with open('W2V_150.txt', 'r', encoding='utf-8') as f:
    N = int(f.readline())
    D = int(f.readline())
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.array(parts[1:], np.float32)
        V[word] = vector

In [3]:
def cosine_similarity(word1, word2):
    if word1 not in V or word2 not in V: return np.nan
    v1, v2 = V[word1], V[word2]
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [4]:
print(cosine_similarity('nóng', 'lạnh'))
print(cosine_similarity('gà', 'vịt'))
print(cosine_similarity('khoai', 'toán'))
print(cosine_similarity('bom', 'internet'))

0.5326895
0.62082016
0.11791506
0.042878658


In [5]:
df = pd.read_csv('Visim-400.txt', sep='\t')
df['Cosine_Sim'] = df.apply(lambda row: cosine_similarity(row['Word1'], row['Word2']), axis=1)
df_clean = df.dropna(subset=['Cosine_Sim'])
pearson, _ = stats.pearsonr(df_clean['Cosine_Sim'], df_clean['Sim2'])
spearman, _ = stats.spearmanr(df_clean['Cosine_Sim'], df_clean['Sim2'])

print(f"Số cặp từ hợp lệ: {len(df_clean)}/{len(df)}")
print(f"Pearson: {pearson:.4f}")
print(f"Spearman: {spearman:.4f}")

Số cặp từ hợp lệ: 344/400
Pearson: 0.4468
Spearman: 0.4078


In [6]:
def k_nearest_words(target_word, k=5):
    similarities = []
    for word, vec in V.items():
        if word == target_word:
            continue
        sim = cosine_similarity(target_word, word)
        similarities.append((word, float(sim)))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:k]

In [7]:
k_nearest_words('nóng')

[('nóng_bỏng', 0.551652193069458),
 ('lạnh', 0.5326895117759705),
 ('làm_nóng', 0.5315719246864319),
 ('nóng_lên', 0.5275053381919861),
 ('hot', 0.5219852924346924)]

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

def extract_features(word1, word2):
  if word1 not in V or word2 not in V:
    return None
  v1, v2 = V[word1], V[word2]
  return np.hstack([v1, v2, np.abs(v1 - v2), v1 * v2])

def load_train_data(syn_path, ant_path):
  X, y = [], []
  for path, label in [(syn_path, 1), (ant_path, 0)]:
    with open(path, "r", encoding="utf-8") as f:
      for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
          feat = extract_features(parts[0], parts[1])
          if feat is not None:
            X.append(feat)
            y.append(label)
  return np.array(X), np.array(y)

def load_test_data(file_paths):
  X, y = [], []
  for path in file_paths:
    df = pd.read_csv(path, sep="\t")
    for _, row in df.iterrows():
      feat = extract_features(str(row["Word1"]), str(row["Word2"]))
      if feat is not None:
        X.append(feat)
        label = 1 if str(row["Relation"]).strip() == "SYN" else 0
        y.append(label)
  return np.array(X), np.array(y)

X_train, y_train = load_train_data("Synonym_vietnamese.txt", "Antonym_vietnamese.txt")

test_files = [
    "400_noun_pairs.txt",
    "400_verb_pairs.txt",
    "600_adj_pairs.txt",
]
X_test, y_test = load_test_data(test_files)

clf = LogisticRegression(max_iter=1000)
# clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print("\nDetailed Report:\n", classification_report(y_test, y_pred, target_names=['Antonym', 'Synonym']))

Precision: 0.7419
Recall:    0.9476
F1-Score:  0.8322

Detailed Report:
               precision    recall  f1-score   support

     Antonym       0.94      0.72      0.82       639
     Synonym       0.74      0.95      0.83       534

    accuracy                           0.83      1173
   macro avg       0.84      0.84      0.83      1173
weighted avg       0.85      0.83      0.83      1173

